# CENTINELA — Panel de operaciones y monitorización

Cuaderno independiente del notebook principal (congelado como anexo del TFM): consume sus
artefactos —el modelo empaquetado y los datos— y construye la herramienta que usaría el
equipo de operaciones. Los meses 6 y 7 se tratan como producción simulada. Produce un panel
interactivo autocontenido (`outputs/dashboard/centinela_ops.html`) y la tabla de deriva
(PSI) por variable.

In [9]:
# Preparación del entorno

from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Configuración compartida del proyecto
%run "/content/drive/MyDrive/TFM_Fraude/config.ipynb"

# Librerías
import zipfile

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Directorios necesarios para este notebook
for ruta in [
    RUTA_LOCAL,
    RUTA_DASH,
] : ruta.mkdir(parents=True, exist_ok=True)
print('Entorno listo.')

Mounted at /content/drive
Entorno listo.


In [10]:
# Carga de datos y del modelo empaquetado
# La función de variables es una copia exacta de la del notebook principal
# y de la API: la misma transformación en entrenamiento y en producción.

if not (RUTA_LOCAL / NOMBRE_CSV).exists():
    with zipfile.ZipFile(RUTA_ZIPS / NOMBRE_ZIP) as z:
        z.extract(NOMBRE_CSV, RUTA_LOCAL)

df = pd.read_csv(RUTA_LOCAL / NOMBRE_CSV, dtype=TIPOS_COLUMNAS)

def construir_features(datos):
    d = datos.copy()
    d['ratio_velocidad_6h_24h'] = (d['velocity_6h'] / d['velocity_24h']).astype('float32')
    d['ratio_velocidad_24h_4w'] = (d['velocity_24h'] / d['velocity_4w']).astype('float32')
    d['presion_zip'] = (d['zip_count_4w'] / d['velocity_4w']).astype('float32')
    for r in ['ratio_velocidad_6h_24h', 'ratio_velocidad_24h_4w', 'presion_zip']:
        d[r] = d[r].replace([np.inf, -np.inf], np.nan)
    d['telefonos_validos'] = (d['phone_home_valid'] + d['phone_mobile_valid']).astype('int8')
    d['similitud_baja'] = (d['name_email_similarity'] < 0.2).astype('int8')
    d['limite_alto'] = (d['proposed_credit_limit'] >= 1500).astype('int8')
    d['dispositivo_multiemail'] = (d['device_distinct_emails_8w'] >= 2).astype('int8')
    d['velocity_6h'] = d['velocity_6h'].clip(lower=0)
    ausencia = {
        'prev_address_months_count': d['prev_address_months_count'] == -1,
        'current_address_months_count': d['current_address_months_count'] == -1,
        'bank_months_count': d['bank_months_count'] == -1,
        'session_length_in_minutes': d['session_length_in_minutes'] == -1,
        'device_distinct_emails_8w': d['device_distinct_emails_8w'] == -1,
        'intended_balcon_amount': d['intended_balcon_amount'] < 0,
    }
    for col, falta in ausencia.items():
        d[col + '_ausente'] = falta.astype('int8')
        d[col] = d[col].astype('float32').where(~falta)
    flags = [c for c in d.columns if c.endswith('_ausente')]
    d['n_faltantes'] = d[flags].sum(axis=1).astype('int8')
    return d

art = joblib.load(RUTA_SALIDAS / 'modelos' / 'centinela_v1.joblib')
UMBRAL = float(art['umbral_operativo'])
C_REV = float(art['coste_revision_$'])
print('Artefacto cargado. Claves:', sorted(art.keys()))
print(f'Umbral de operación: {UMBRAL:.4f} | Coste de revisión: {C_REV:.0f} $')

Artefacto cargado. Claves: ['calibrador', 'categoricas', 'columnas', 'coste_revision_$', 'esquema_temporal', 'fecha_entrenamiento', 'metricas_test', 'modelo', 'numericas', 'preprocesador', 'umbral_operativo']
Umbral de operación: 0.0330 | Coste de revisión: 10 $


In [11]:
# Scoring de la producción simulada (meses 6 y 7)
# El panel usa exactamente el mismo camino que la API: variables, preprocesado,
# modelo y calibrador del artefacto. El ahorro calculado aquí debe coincidir
# con el del notebook principal: es la comprobación de integridad del sistema.

prod = df[df['month'] >= 6]
Xp = construir_features(prod)[art['columnas']]
p = art['calibrador'].predict(
    art['modelo'].predict_proba(art['preprocesador'].transform(Xp))[:, 1])

y = prod['fraud_bool'].to_numpy()
lim = prod['proposed_credit_limit'].to_numpy()
alerta = p >= UMBRAL
DIAS_TEST = 61

perdida_sin_modelo = lim[y == 1].sum()
ahorro = perdida_sin_modelo - (C_REV * alerta.sum() + lim[(y == 1) & (~alerta)].sum())
recall = 100 * alerta[y == 1].mean()

resumen = prod.assign(alerta=alerta).groupby('month').apply(
    lambda g: pd.Series({
        'solicitudes': len(g),
        'alertas': int(g['alerta'].sum()),
        'tasa_fraude_%': 100 * g['fraud_bool'].mean(),
        'ahorro_$': g.loc[g['fraud_bool'] == 1, 'proposed_credit_limit'].sum()
                    - (C_REV * g['alerta'].sum()
                       + g.loc[(g['fraud_bool'] == 1) & (~g['alerta']),
                               'proposed_credit_limit'].sum()),
    }), include_groups=False)
print(resumen.round(1).to_string())
print(f'\nAhorro total: {ahorro:,.0f} $ | Recall: {recall:.1f}% | '
      f'Alertas/día: {alerta.sum() / DIAS_TEST:,.0f}')
print('Debe coincidir con el notebook principal: 1,558,270 $')

       solicitudes  alertas  tasa_fraude_%  ahorro_$
month                                               
6         108168.0  12461.0            1.3  771660.0
7          96843.0   8124.0            1.5  786610.0

Ahorro total: 1,558,270 $ | Recall: 66.0% | Alertas/día: 337
Debe coincidir con el notebook principal: 1,558,270 $


In [12]:
# Monitor de deriva: PSI por variable
# El PSI (Population Stability Index) compara la distribución de cada variable
# en producción contra la del periodo de entrenamiento. Lectura bancaria
# habitual: < 0,10 estable | 0,10-0,25 vigilar | > 0,25 alerta.

def psi(base, actual, bins=10):
    base = base.dropna(); actual = actual.dropna()
    if str(base.dtype) in ('category', 'object', 'str') or base.nunique() <= 10:
        p_ = base.astype('object').value_counts(normalize=True)
        q_ = actual.astype('object').value_counts(normalize=True).reindex(p_.index).fillna(0)
    else:
        bordes = np.unique(np.quantile(base, np.linspace(0, 1, bins + 1)))
        bordes[0], bordes[-1] = -np.inf, np.inf
        p_ = pd.cut(base, bordes).value_counts(normalize=True).sort_index()
        q_ = pd.cut(actual, bordes).value_counts(normalize=True).sort_index()
    p_ = p_.clip(lower=1e-6); q_ = q_.clip(lower=1e-6)
    return float(((p_ - q_) * np.log(p_ / q_)).sum())

referencia = df[df['month'] <= 5]
variables = [c for c in df.columns if c not in ('fraud_bool', 'month', 'device_fraud_count')]
tabla_psi = pd.DataFrame({
    f'mes_{m}': {v: psi(referencia[v], df.loc[df['month'] == m, v]) for v in variables}
    for m in [6, 7]
}).sort_values('mes_7', ascending=False)

print(tabla_psi.head(12).round(3).to_string())
print(f"\nEn alerta (>0,25) en el mes 7: {int((tabla_psi['mes_7'] > 0.25).sum())} variables "
      f"| PSI mediano: {tabla_psi['mes_7'].median():.3f}")
tabla_psi.to_csv(RUTA_SALIDAS / 'reportes' / 'monitor_psi.csv')

# Las velocidades globales del sistema encabezan la deriva (velocity_4w supera
# un PSI de 3, un orden de magnitud sobre el umbral de alerta) mientras la
# mediana del resto es ~0,04: el entorno cambió, no los solicitantes. Sumado al
# aumento de la prevalencia (1,03% -> 1,48%), el monitor habría recomendado
# reentrenar ya durante el primer mes de despliegue.

                                  mes_6  mes_7
velocity_4w                       2.696  3.698
velocity_24h                      1.282  2.323
velocity_6h                       0.743  1.225
zip_count_4w                      0.231  0.787
date_of_birth_distinct_emails_4w  0.300  0.489
credit_risk_score                 0.205  0.244
bank_branch_count_8w              0.016  0.093
current_address_months_count      0.020  0.091
session_length_in_minutes         0.015  0.088
income                            0.079  0.063
proposed_credit_limit             0.084  0.050
customer_age                      0.010  0.045

En alerta (>0,25) en el mes 7: 5 variables | PSI mediano: 0.040


In [13]:
# Ensamblaje del panel y exportación a HTML autocontenido
# Las variables del monitor se muestran con nombres cortos de negocio para
# que las etiquetas no invadan el gráfico vecino.

fig = make_subplots(
    rows=3, cols=4, row_heights=[0.16, 0.42, 0.42],
    vertical_spacing=0.10, horizontal_spacing=0.14,
    specs=[[{'type': 'domain'}] * 4,
           [{'colspan': 2, 'secondary_y': True}, None, {'colspan': 2}, None],
           [{'colspan': 2}, None, {'colspan': 2}, None]],
    subplot_titles=[None, None, None, None,
                    'Volumen y tasa de fraude por mes',
                    'Deriva de variables — PSI mes 7 (top 10)',
                    'Probabilidad calibrada por clase (eje log)',
                    'Ahorro frente al umbral (línea: operación)'])

indicadores = [
    ('Ahorro test (2 meses)', ahorro, ',.0f', ' $'),
    ('Umbral de operación', UMBRAL, '.4f', ''),
    ('Alertas/día observadas', alerta.sum() / DIAS_TEST, ',.0f', ''),
    ('Recall en operación', recall, '.1f', ' %'),
]
for i, (titulo, valor, formato, sufijo) in enumerate(indicadores, start=1):
    fig.add_trace(go.Indicator(
        mode='number', value=valor,
        title={'text': titulo, 'font': {'size': 13}},
        number={'valueformat': formato, 'suffix': sufijo, 'font': {'size': 28}}),
        row=1, col=i)

m = resumen.reset_index()
fig.add_trace(go.Bar(x=m['month'], y=m['solicitudes'], name='Solicitudes',
                     marker_color=COLOR_LEGITIMO), row=2, col=1)
fig.add_trace(go.Scatter(x=m['month'], y=m['tasa_fraude_%'], name='Tasa de fraude %',
                         mode='lines+markers', marker_color=COLOR_FRAUDE),
              row=2, col=1, secondary_y=True)
fig.update_xaxes(tickvals=[6, 7], ticktext=['Mes 6', 'Mes 7'], row=2, col=1)

top10 = tabla_psi['mes_7'].head(10).iloc[::-1]
etiquetas = [NOMBRES_CORTOS.get(v, v[:20]) for v in top10.index]
colores = [COLOR_FRAUDE if v > 0.25 else COLOR_AVISO if v > 0.10 else COLOR_LEGITIMO
           for v in top10]
fig.add_trace(go.Bar(x=top10.values, y=etiquetas, orientation='h',
                     marker_color=colores, showlegend=False), row=2, col=3)
fig.update_yaxes(tickfont={'size': 11}, row=2, col=3)
fig.update_xaxes(title_text='PSI', title_font={'size': 11}, row=2, col=3)

bordes = np.linspace(0, 1, 61)
centros = (bordes[:-1] + bordes[1:]) / 2
for clase, nombre, color in [(0, 'Legítimas', COLOR_LEGITIMO), (1, 'Fraudes', COLOR_FRAUDE)]:
    conteo, _ = np.histogram(p[y == clase], bins=bordes)
    fig.add_trace(go.Bar(x=centros, y=conteo, name=nombre, marker_color=color,
                         opacity=0.7, width=bordes[1] - bordes[0]), row=3, col=1)
fig.update_yaxes(type='log', row=3, col=1)

umbrales = np.quantile(p, np.linspace(0.001, 0.999, 200))
curva = np.array([perdida_sin_modelo - (C_REV * (p >= u).sum() + lim[(y == 1) & (p < u)].sum())
                  for u in umbrales]) / 1000
fig.add_trace(go.Scatter(x=umbrales, y=curva, mode='lines', name='Ahorro (miles $)',
                         line_color=COLOR_LEGITIMO), row=3, col=3)
fig.add_trace(go.Scatter(x=[UMBRAL, UMBRAL], y=[float(curva.min()), float(curva.max())],
                         mode='lines', name='Umbral de operación',
                         line=dict(color=COLOR_FRAUDE, dash='dash')), row=3, col=3)

fig.update_layout(template='plotly_white', height=940, barmode='overlay',
                  title_text='CENTINELA — Panel de Operaciones y Monitorización '
                             '(meses 6-7 como producción simulada)',
                  legend=dict(orientation='h', y=-0.07))
ruta_html = RUTA_DASH / 'centinela_ops.html'
fig.write_html(ruta_html, include_plotlyjs=True, full_html=True)
print(f'Panel exportado: {ruta_html} '
      f'({ruta_html.stat().st_size / 1_048_576:.1f} MB)')
fig.show()

Panel exportado: /content/drive/MyDrive/TFM_Fraude/outputs/dashboard/centinela_ops.html (4.4 MB)


### Lectura del panel

La fila de indicadores resume la operación del periodo. El panel superior izquierdo muestra
la caída de volumen junto al ascenso de la tasa de fraude; el derecho, el semáforo de deriva:
las velocidades globales del sistema superan con holgura el umbral de alerta (PSI > 0,25)
mientras el resto de variables permanece estable. Abajo, la separación entre clases que
consigue la probabilidad calibrada y la curva de ahorro con el umbral de operación marcado.

**Uso:** el archivo `centinela_ops.html` es autocontenido — se descarga de
`outputs/dashboard/` y se abre en cualquier navegador sin instalar nada. Sirve como material
interactivo para los tutores y como toma de pantalla para el vídeo.

In [14]:
# Exportación de datos agregados para el cuadro de mando ejecutivo (Power BI)
# Se exportan únicamente agregados y métricas derivadas, nunca filas del
# dataset original: así el archivo .pbix respeta la licencia CC BY-NC-ND.
# Nota aprendida: $ no puede formar parte de un nombre de argumento en Python,
# solo de textos entre comillas; por eso la columna se crea y luego se renombra.

RUTA_PBI = RUTA_SALIDAS / 'powerbi'
RUTA_PBI.mkdir(exist_ok=True)

# 1. Indicadores generales del sistema (una fila)
pd.DataFrame([{
    'perdida_sin_modelo_$': float(perdida_sin_modelo),
    'ahorro_$': float(ahorro),
    'umbral_operacion': UMBRAL,
    'alertas_dia': float(alerta.sum() / DIAS_TEST),
    'recall_pct': float(recall),
    'coste_revision_$': C_REV,
}]).to_csv(RUTA_PBI / 'kpis.csv', index=False)

# 2. Evolución mensual completa (histórico + producción)
evolucion = df.groupby('month').agg(
    solicitudes=('fraud_bool', 'size'),
    fraudes=('fraud_bool', 'sum'),
    exposicion_fraude=('proposed_credit_limit',
                       lambda s: s[df.loc[s.index, 'fraud_bool'] == 1].sum()),
).reset_index()
evolucion['tasa_fraude_pct'] = 100 * evolucion['fraudes'] / evolucion['solicitudes']
evolucion['periodo'] = np.where(evolucion['month'] <= 5, 'Entrenamiento', 'Producción')
evolucion = evolucion.rename(columns={'exposicion_fraude': 'exposicion_fraude_$'})
evolucion.to_csv(RUTA_PBI / 'evolucion_mensual.csv', index=False)

# 3. Operación en producción por mes (alertas y ahorro)
resumen.reset_index().to_csv(RUTA_PBI / 'operacion_mensual.csv', index=False)

# 4. Monitor de deriva con semáforo
psi_pbi = tabla_psi.reset_index().rename(columns={'index': 'variable'})
psi_pbi['estado'] = pd.cut(psi_pbi['mes_7'], [-1, 0.10, 0.25, np.inf],
                           labels=['Estable', 'Vigilar', 'Alerta'])
psi_pbi.to_csv(RUTA_PBI / 'monitor_psi.csv', index=False)

# 5. Distribución de la probabilidad calibrada (pre-agregada en 60 tramos)
bordes = np.linspace(0, 1, 61)
centros = (bordes[:-1] + bordes[1:]) / 2
dist = []
for clase, nombre in [(0, 'Legítima'), (1, 'Fraude')]:
    conteo, _ = np.histogram(p[y == clase], bins=bordes)
    dist += [{'probabilidad': c, 'clase': nombre, 'solicitudes': int(n)}
             for c, n in zip(centros, conteo)]
pd.DataFrame(dist).to_csv(RUTA_PBI / 'distribucion_scores.csv', index=False)

# 6. Curva de ahorro por umbral
pd.DataFrame({'umbral': umbrales, 'ahorro_$': curva * 1000}).to_csv(
    RUTA_PBI / 'curva_ahorro.csv', index=False)

# 7. Trato por grupo de edad en el punto de operación
grupo = prod['customer_age'].to_numpy() >= 50
filas = []
for nombre, m in [('50 años o más', grupo), ('Menos de 50', ~grupo)]:
    yl, al = y[m], alerta[m]
    filas.append({'grupo': nombre, 'solicitudes': int(m.sum()),
                  'FPR_pct': 100 * al[yl == 0].mean(),
                  'TPR_pct': 100 * al[yl == 1].mean()})
pd.DataFrame(filas).to_csv(RUTA_PBI / 'fairness_grupos.csv', index=False)

print('Exportados 7 archivos a', RUTA_PBI)
for f in sorted(RUTA_PBI.glob('*.csv')):
    print(f'  - {f.name}')

Exportados 7 archivos a /content/drive/MyDrive/TFM_Fraude/outputs/powerbi
  - curva_ahorro.csv
  - distribucion_scores.csv
  - evolucion_mensual.csv
  - fairness_grupos.csv
  - kpis.csv
  - monitor_psi.csv
  - operacion_mensual.csv
